In [3]:
%pip -q install youtube-transcript-api pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from youtube_transcript_api import YouTubeTranscriptApi
import pandas as pd
from IPython.display import FileLink, display
import os

In [5]:
VIDEO_ID = "wzssm02u35I" 

In [6]:
def fetch_transcript_to_csv(video_id):
    """Fetch transcript and save to CSV"""
    try:
        # Fetch transcript using current API (v1.2+)
        api_client = YouTubeTranscriptApi()
        segments_list = api_client.fetch(video_id).to_raw_data()
        
        # Convert to DataFrame
        df = pd.DataFrame(segments_list)
        df['video_id'] = video_id
        
        # Reorder columns
        df = df[['video_id', 'start', 'duration', 'text']]
        
        # Save to CSV
        csv_path = f"{video_id}_transcript.csv"
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")
        
        print(f"Fetched segments: {len(df)}")
        print(f"Saved to: {os.path.abspath(csv_path)}")
        display(FileLink(csv_path))
        
        return df
    
    except Exception as e:
        print(f"Error: {e}")
        return None

# Fetch and save transcript
transcript_df = fetch_transcript_to_csv(VIDEO_ID)
if transcript_df is not None:
    print("\nFirst 5 segments:")
    print(transcript_df.head())

Fetched segments: 918
Saved to: d:\Agentic AI\wzssm02u35I_transcript.csv


d:\Agentic AI\wzssm02u35I_transcript.csv


First 5 segments:
      video_id  start  duration                                      text
0  wzssm02u35I  0.400     3.760        use this information to answer the
1  wzssm02u35I  2.320     3.920    question. We are also sending the data
2  wzssm02u35I  4.160     3.759  or the context along with it. Now I need
3  wzssm02u35I  6.240     3.200             to scrape my site and add the
4  wzssm02u35I  7.919     3.840       information in the vector database.


In [7]:

import os
os.environ["GROQ_API_KEY"] = "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"

In [8]:
%pip install langchain_community langchainhub chromadb langchain langchain-openai beautifulsoup4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path="wzssm02u35I_transcript.csv",
    encoding="utf-8"  # Specify UTF-8 encoding
)
docs = loader.load()

print(f"Loaded {len(docs)} documents from CSV")
print("\nFirst document:")
print(docs[0])

Loaded 918 documents from CSV

First document:
page_content='﻿video_id: wzssm02u35I
start: 0.4
duration: 3.76
text: use this information to answer the' metadata={'source': 'wzssm02u35I_transcript.csv', 'row': 0}


In [11]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Use larger chunks so a whole thread (question + replies) stays together
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 4000, chunk_overlap = 200)
splits = text_splitter.split_documents(docs)

In [12]:
print(splits[0])
print(splits[1])
print(splits[2])

page_content='﻿video_id: wzssm02u35I
start: 0.4
duration: 3.76
text: use this information to answer the' metadata={'source': 'wzssm02u35I_transcript.csv', 'row': 0}
page_content='﻿video_id: wzssm02u35I
start: 2.32
duration: 3.92
text: question. We are also sending the data' metadata={'source': 'wzssm02u35I_transcript.csv', 'row': 1}
page_content='﻿video_id: wzssm02u35I
start: 4.16
duration: 3.759
text: or the context along with it. Now I need' metadata={'source': 'wzssm02u35I_transcript.csv', 'row': 2}


In [13]:
pip install langchain-groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from langchain_groq import ChatGroq

In [15]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Fast, small embedding model
hf_emb = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# Rebuild store from threaded, larger chunks
vectorstore = Chroma.from_documents(documents=splits, embedding=hf_emb)


In [17]:
print(vectorstore._collection.get())

{'ids': ['0b932cb9-888d-4018-94fc-26df0ab21f9e', 'c558ccbf-dc9d-41a1-84d1-2d6058639270', '1b4a3657-5de0-4866-9a8e-ac6cc8611d18', '411578b6-39ae-4bd2-95d9-ae6841778ced', '89d8df73-b533-4c12-a50c-76af5325d828', 'ca2a5a89-e9fe-4f04-9dca-4f660f85227a', '090a830f-07c0-4332-8637-485c5b7aabc3', '97e8bbd1-e8b1-4955-bbc9-835baed72031', 'b8d72e6c-1c95-4d4a-b334-d0bbad88a431', '16c0c5f1-4031-4d22-b9bd-9b3ca3d2b502', '31892ec9-69ad-45e8-81ed-8544a5528912', 'b246ed9c-666c-462a-b065-689e8af1c37a', 'd99be0d8-4fe4-48fe-97ff-8a902bf2d7d9', '8a4a804c-b52a-41c9-856c-bd7b280b4ac0', '30dcdd85-7c8e-41eb-9eac-b8b7912e3cdf', '8cbe1db1-cbe6-4e30-8acb-c68b35420878', '8ca9d56c-c9a0-4083-8913-065d14cd761d', 'dc27a899-33bb-43f6-8327-3c22b858692f', '9712de8e-16be-4162-aa28-2400b120eac9', '262a6305-4c02-48d8-9229-133292052bf3', 'd4fbe0fc-1d3d-480c-adba-dd461da5e4ac', 'b09d033d-cbfc-44d1-84fe-f64c35ba6a43', 'dbdd5584-60d2-4d70-a6d3-7fa9f35d2bc2', '6ebc0943-701b-447d-b6f7-c3ecf82202bb', '4ed34dda-cbe7-4347-948b-89e930

In [18]:
print("\nCollection 1 - ", vectorstore._collection.get(ids=['28651d9a-ab51-41f8-ab83-e68285623c4e'], include=["embeddings", "documents"]))
print("\nCollection 2 - ", vectorstore._collection.get(ids=['054dee19-19ed-4574-bc51-511060fd707a'], include=["embeddings", "documents"]))
print("\nCollection 3 - ", vectorstore._collection.get(ids=['2fd71cb4-835a-43c5-b920-b7e1be51c450'], include=["embeddings", "documents"]))


Collection 1 -  {'ids': [], 'embeddings': array([], dtype=float64), 'documents': [], 'uris': None, 'included': ['embeddings', 'documents'], 'data': None, 'metadatas': None}

Collection 2 -  {'ids': [], 'embeddings': array([], dtype=float64), 'documents': [], 'uris': None, 'included': ['embeddings', 'documents'], 'data': None, 'metadatas': None}

Collection 3 -  {'ids': [], 'embeddings': array([], dtype=float64), 'documents': [], 'uris': None, 'included': ['embeddings', 'documents'], 'data': None, 'metadatas': None}


In [20]:
from langchain_core.runnables import RunnableLambda

def simple_retrieve(query: str):
    """Simple MMR-based retrieval for transcript chunks"""
    # Use MMR (Maximal Marginal Relevance) for diverse, relevant results
    docs = vectorstore.max_marginal_relevance_search(
        query, 
        k=5,           # Return top 5 chunks
        fetch_k=20,    # Consider top 20 for diversity
        lambda_mult=0.5  # Balance relevance vs diversity
    )
    return docs

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5}
)

In [21]:
from langchain_core.prompts import ChatPromptTemplate

system_instructions = (
    "You are an assistant that answers strictly and only using the provided YouTube transcript. "
    "Do not add outside knowledge. "
    "Always cite exact timestamps for each claim using the format [timestamp]. "
    "If the answer is not in the transcript, say 'Not found in transcript.'"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_instructions),
    ("human", (
        "Question:\\n{question}\\n\\n"
        "Transcript context (subset):\\n{context}\\n\\n"
        "Instruction:\\n"
        "- Identify the relevant time segments that contain information about the question.\\n"
        "- Extract key points, explanations, or examples from the transcript.\\n"
        "- Respond with clear, concise answers.\\n"
        "- Cite each point with [timestamp]."
    ))
])

In [22]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", api_key="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")

In [23]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [24]:
def format_docs(docs):
    """Format retrieved documents for context"""
    return "\\n\\n".join(doc.page_content for doc in docs)

In [25]:
dag = ({"context" : retriever | format_docs, "question" : RunnablePassthrough()}
       | prompt
       | llm
       | StrOutputParser())

rag_chain = dag

In [27]:
print(rag_chain.invoke("what is rag model"))

**What is a RAG model?**  
The transcript indicates that a RAG (Retrieval‑Augmented Generation) model is built around a *retriever* that pulls out relevant text chunks and passes them to a language model (LLM). The LLM then uses those retrieved chunks to generate its output.  
- The concept behind RAG is mentioned as a key idea in the video [101.6].  
- It is described as part of a “lang chain rag” system that incorporates retrieval and generation steps [244.239].  
- The structure is further clarified as a new chain where the retriever extracts chunks and hands them to the LLM, which still performs the final generation [1817.679, 1973.679, 1895.44].  

If you need more detail, the transcript does not provide further explanation.
